# Service Ticket Priority Classification

Minimal implementation: definitions (with comments), training, prediction.

In [ ]:
# ---------------------------------------------------------------------------
# Shared numeric/pandas imports and one date helper
# ---------------------------------------------------------------------------
import numpy as np  # numeric cleanup, infinities, probability arrays
import pandas as pd  # CSV loading, cleaning, feature engineering


def _dates(values):
    # Parse ISO dates first, then day-first dates; unreadable text becomes NaT.
    s = values.astype('string')  # string dtype keeps missing values as <NA>
    parsed = pd.to_datetime(s, errors='coerce', format='%Y-%m-%d %H:%M')
    missing = parsed.isna()  # ISO format could not read these rows
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(s[missing], errors='coerce', dayfirst=True)
    return parsed


# Only the imports this task actually uses.
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ---------------------------------------------------------------------------
# Feature contracts: exact column groups the tests check
# ---------------------------------------------------------------------------
RAW_NUMERIC = ['response_minutes', 'touch_count', 'backlog_count',
               'agent_count', 'agent_experience_yrs']  # 5 original measurements
DERIVED_NUMERIC = ['opened_hour', 'opened_month', 'backlog_per_agent']  # 3 calculated
CATEGORICAL = ['channel', 'product_area', 'customer_tier']  # 3 text/coded fields
NUMERIC_FEATURES = RAW_NUMERIC + DERIVED_NUMERIC  # 8 numeric predictors
FEATURES = NUMERIC_FEATURES + CATEGORICAL  # exactly 11 model inputs
TARGET = 'priority'  # label column, never a predictor
CLASS_NAMES = ['HIGH', 'LOW', 'MEDIUM']  # fixed class order -> encoder gives 0,1,2


def load_data(filename):
    # A1: clean raw CSV and add features; same code for train, test and predict files.
    df = pd.read_csv(filename).drop_duplicates('ticket_id', keep='first').copy()
    # Duplicate IDs must not leak into training/evaluation; keep first occurrence.
    df['opened_ts'] = _dates(df['opened_ts'])
    df = df[df['opened_ts'].notna()].copy()  # invalid timestamps cannot produce time features
    for c in RAW_NUMERIC:
        # numeric-looking text -> number; inf -> NaN so imputation can handle it
        df[c] = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
    for c in CATEGORICAL:
        # strip/lowercase, turn "2.0" into "2", blanks -> "unknown"
        df[c] = (df[c].astype('string').str.strip().str.lower()
                 .str.replace(r'\.0$', '', regex=True).replace('', pd.NA).fillna('unknown'))
    df['opened_hour'] = df['opened_ts'].dt.hour  # hour is useful for workload patterns
    df['opened_month'] = df['opened_ts'].dt.month  # month adds seasonality
    # Division is protected: zero/negative agent counts become NaN, never inf.
    df['backlog_per_agent'] = df['backlog_count'] / df['agent_count'].where(df['agent_count'] > 0)
    if TARGET in df.columns:
        # Normalize labels: " high " -> "HIGH", blank -> missing so bad rows can be filtered.
        df[TARGET] = df[TARGET].astype('string').str.strip().str.upper().replace('', pd.NA)
    return df


def build_pipeline(algorithm):
    # A2: reusable preprocessing + estimator; fitted separately for each model.
    preprocessor = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  # median resists outliers
            ('scaler', StandardScaler())  # scaling helps logistic regression
        ]), NUMERIC_FEATURES),

        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))  # unseen categories -> zeros
        ]), CATEGORICAL)
    ])  # two independent branches, as required
    return Pipeline([('preprocessor', preprocessor), ('model', algorithm)])


def prepare_datasets(train_file, test_file):
    train = load_data(train_file)
    test = load_data(test_file)

    train = train[train[TARGET].isin(CLASS_NAMES)]
    test = test[test[TARGET].isin(CLASS_NAMES)]

    encoder = LabelEncoder().fit(CLASS_NAMES)

    X_train = train[FEATURES]
    X_test = test[FEATURES]

    y_train = pd.Series(encoder.transform(train[TARGET]), index=train.index)
    y_test = pd.Series(encoder.transform(test[TARGET]), index=test.index)

    return X_train, X_test, y_train, y_test, encoder


def train_models(X_train, y_train):
    # 2: fit both candidate classifiers; balanced weights help unequal class sizes.
    models = {
        'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'random_forest': RandomForestClassifier(n_estimators=80, max_depth=10, min_samples_leaf=2,
                                                class_weight='balanced', random_state=42, n_jobs=-1),
    }
    for name, algo in models.items():
        models[name] = build_pipeline(algo).fit(X_train, y_train)  # preprocessing fitted on train only
    return models


def evaluate_models(trained_models, X_test, y_test):
    # 3: score each fitted model on held-back data; never refit here.
    rows, matrices = [], {}
    for name, model in trained_models.items():
        pred = model.predict(X_test)
        rows.append({'model_name': name,
                     'accuracy': accuracy_score(y_test, pred),
                     'macro_f1': f1_score(y_test, pred, labels=[0, 1, 2], average='macro', zero_division=0),
                     'weighted_f1': f1_score(y_test, pred, labels=[0, 1, 2], average='weighted', zero_division=0)})
        matrices[name] = confusion_matrix(y_test, pred, labels=[0, 1, 2])  # fixed 3x3 layout
    comparison = (pd.DataFrame(rows)
                  .sort_values(['macro_f1', 'weighted_f1', 'model_name'],
                               ascending=[False, False, True])  # ranking rule from instructions
                  .reset_index(drop=True))
    return comparison, matrices


def select_best_model(comparison, trained_models):
    # 4: top row is already ranked; return that existing pipeline, not a copy.
    name = comparison.iloc[0]['model_name']
    return name, trained_models[name]


def predict_priority(model, encoder, df):
    # 5: one probability pass gives both the chosen class and its confidence.
    proba = model.predict_proba(df[FEATURES])
    pred = model.classes_[proba.argmax(axis=1)]  # argmax aligns with class order
    high_col = list(model.classes_).index(0)  # class 0 is HIGH in our encoder
    result = pd.DataFrame({
        'ticket_id': df['ticket_id'].to_numpy(),  # preserve ID and row order
        'predicted_priority': encoder.inverse_transform(pred),  # readable HIGH/LOW/MEDIUM
        'confidence': np.round(proba.max(axis=1), 4),  # probability of chosen class
        'high_probability': np.round(proba[:, high_col], 4),  # probability of HIGH
    })
    return result


def train():
    # TRAINING WORKFLOW: the only cell that fits models and writes artifacts.
    X_train, X_test, y_train, y_test, encoder = prepare_datasets('ticket_train.csv', 'ticket_test.csv')
    models = train_models(X_train, y_train)
    comparison, matrices = evaluate_models(models, X_test, y_test)
    name, best = select_best_model(comparison, models)
    joblib.dump({'model': best, 'encoder': encoder}, 'best_model.pkl')  # model + encoder together
    comparison.to_csv('model_comparison.csv', index=False)
    pd.DataFrame(matrices[name], index=CLASS_NAMES, columns=CLASS_NAMES).to_csv('confusion_matrix.csv')
    return name, comparison


def predict():
    # PREDICTION WORKFLOW: reload saved artifacts; no fitting, no source CSV changes.
    saved = joblib.load('best_model.pkl')
    result = predict_priority(saved['model'], saved['encoder'], load_data('ticket_predict.csv'))
    result.to_csv('ticket_predictions.csv', index=False)
    return result


In [ ]:
# TRAINING WORKFLOW: this is the only cell that fits models and writes artifacts.
name, comparison = train()
comparison

In [ ]:
# PREDICTION WORKFLOW: reload the saved model; no fitting happens here.
predict()